# Appendix 9. DVC 기초

## 1. Goal

이 notebook은 DVC의 data tracking과 pipeline 재현 흐름을 작은 임시 project에서 익히는 자료입니다. `dvc add`, `dvc stage add`, `dvc repro`, `dvc status`, `dvc dag`를 실행하고 `.dvc` file, `dvc.yaml`, `params.yaml`, `dvc.lock`이 각각 무엇을 기록하는지 확인합니다.

DVC는 Git을 대체하지 않습니다. Git이 code와 작은 metadata file의 revision을 관리하고, DVC는 큰 data와 pipeline output의 cache·remote·lineage를 연결합니다.

## 2. Setup

프로젝트에 설치된 DVC 3.x CLI와 Git을 사용합니다. 현재 repository의 `dvc.yaml`, `dvc.lock`, data와 cache는 건드리지 않으며, 모든 write command는 임시 디렉터리에서 실행하고 마지막 Checks에서 정리합니다. remote storage나 credential은 사용하지 않습니다.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

import dvc
import yaml

print({"dvc": dvc.__version__})


## 3. Steps

### 3-1. DVC project와 data tracking

#### 3-1-1. 임시 Git·DVC project 초기화

`dvc init`은 `.dvc/` 설정과 Git이 추적할 DVC metadata를 만듭니다. 아래 helper는 command, exit code와 bounded output을 한곳에서 다루고 실패 시 즉시 예외를 발생시킵니다. 실제 project에서는 repository마다 한 번만 초기화합니다.

In [ ]:
temporary_directory = tempfile.TemporaryDirectory(prefix="dvc-appendix-")
project_root = Path(temporary_directory.name)


def run_command(*command: str) -> subprocess.CompletedProcess[str]:
    """Run one bounded command inside the temporary tutorial project."""
    return subprocess.run(
        command,
        cwd=project_root,
        check=True,
        capture_output=True,
        text=True,
    )


def run_dvc(*arguments: str) -> subprocess.CompletedProcess[str]:
    """Run one DVC CLI command inside the temporary tutorial project."""
    return run_command("dvc", *arguments)


run_command("git", "init", "--quiet")
run_dvc("init", "--quiet")
print(sorted(path.name for path in project_root.iterdir()))


#### 3-1-2. dvc add와 pointer metadata

`dvc add`는 data content를 cache에 보관하고 `<name>.dvc` pointer file에 hash, size와 path를 기록합니다. Git에는 원본 data 대신 `.dvc` file과 `.gitignore`를 commit합니다. `dvc push`와 `dvc pull`은 local cache와 구성한 remote storage 사이에서 content를 전송합니다.

In [ ]:
dataset_path = project_root / "dataset.csv"
dataset_path.write_text("record_id,value\nP101,7\nP102,9\n", encoding="utf-8")
run_dvc("add", "dataset.csv")

pointer_path = project_root / "dataset.csv.dvc"
pointer_document = yaml.safe_load(pointer_path.read_text(encoding="utf-8"))
pointer_output = pointer_document["outs"][0]
print(
    {
        "path": pointer_output["path"],
        "hash_prefix": pointer_output["md5"][:8],
        "size": pointer_output["size"],
    }
)


### 3-2. Pipeline stage와 재현

#### 3-2-1. command, dependency, parameter와 output 선언

DVC stage는 실행할 command(`cmd`), 입력 file이나 code(`deps`), versioned parameter(`params`), 생성 결과(`outs`)를 선언합니다. stage command가 실제로 읽는 값만 dependency로 적어야 변경 영향 범위를 정확히 계산할 수 있습니다. 아래 stage는 숫자 하나와 multiplier로 결과 file을 만듭니다.

In [ ]:
(project_root / "source.txt").write_text("7\n", encoding="utf-8")
(project_root / "params.yaml").write_text(
    "prepare:\n  multiplier: 3\n", encoding="utf-8"
)
(project_root / "prepare.py").write_text(
    "from pathlib import Path\n"
    "import yaml\n"
    "params = yaml.safe_load(Path('params.yaml').read_text())\n"
    "value = int(Path('source.txt').read_text())\n"
    "result = value * params['prepare']['multiplier']\n"
    "Path('result.txt').write_text(f'{result}\\n')\n",
    encoding="utf-8",
)
run_dvc(
    "stage",
    "add",
    "--name",
    "prepare",
    "--deps",
    "prepare.py",
    "--deps",
    "source.txt",
    "--params",
    "prepare.multiplier",
    "--outs",
    "result.txt",
    "python",
    "prepare.py",
)
pipeline_document = yaml.safe_load(
    (project_root / "dvc.yaml").read_text(encoding="utf-8")
)
pipeline_document


#### 3-2-2. dvc repro와 dvc.lock

`dvc repro`는 dependency graph와 현재 workspace를 비교해 필요한 stage만 실행합니다. 성공하면 `dvc.lock`에 실제 command, dependency와 output hash, 사용한 parameter 값을 기록합니다. `dvc.yaml`이 재현 recipe라면 `dvc.lock`은 한 번 재현된 구체적인 상태입니다. 둘 다 Git revision에 포함해야 provenance를 따라갈 수 있습니다.

In [ ]:
first_repro = run_dvc("repro")
first_result = int((project_root / "result.txt").read_text(encoding="utf-8"))
lock_document = yaml.safe_load(
    (project_root / "dvc.lock").read_text(encoding="utf-8")
)
locked_stage = lock_document["stages"]["prepare"]
print(
    {
        "result": first_result,
        "locked_parameter": locked_stage["params"]["params.yaml"][
            "prepare.multiplier"
        ],
        "dependency_count": len(locked_stage["deps"]),
        "output_count": len(locked_stage["outs"]),
    }
)


### 3-3. 변경 감지와 dependency graph

#### 3-3-1. dvc status로 stale stage 확인

`dvc status`는 workspace를 `dvc.lock` 상태와 비교합니다. parameter나 dependency가 바뀌면 영향을 받는 stage를 stale로 표시합니다. 변경을 발견했다고 자동으로 실행하지는 않으므로, 결과를 검토한 뒤 `dvc repro`로 필요한 stage를 재현합니다.

In [ ]:
clean_status = run_dvc("status").stdout.strip()
(project_root / "params.yaml").write_text(
    "prepare:\n  multiplier: 5\n", encoding="utf-8"
)
stale_status = run_dvc("status").stdout.strip()
print({"clean": clean_status, "after_parameter_change": stale_status})


#### 3-3-2. dvc dag와 선택적 재현

`dvc dag`는 stage와 dependency 관계를 보여줍니다. 여러 stage가 연결된 project에서는 변경된 dependency의 downstream stage만 다시 실행됩니다. stage 이름을 `dvc repro <stage>`에 넘기면 특정 target까지 선택적으로 재현할 수 있습니다.

In [ ]:
dag_output = run_dvc("dag").stdout.strip()
second_repro = run_dvc("repro", "prepare")
second_result = int((project_root / "result.txt").read_text(encoding="utf-8"))
final_status = run_dvc("status").stdout.strip()
print(
    {
        "dag": dag_output,
        "result_after_parameter_change": second_result,
        "final_status": final_status,
    }
)


#### 3-3-3. cache, remote와 Git의 역할

local cache는 같은 content를 다시 만들거나 받을 때 재사용합니다. remote는 팀이나 다른 환경이 cache content를 공유하는 저장소이며 `dvc remote add`, `dvc push`, `dvc pull`로 연결합니다. Git에는 `.dvc` pointer, `dvc.yaml`, `params.yaml`, `dvc.lock`과 source code를 기록합니다. credential은 private config나 runtime secret으로 관리하고 repository에 넣지 않습니다.

In [ ]:
remote_list = run_dvc("remote", "list").stdout.strip()
cache_directory = project_root / ".dvc" / "cache"
print(
    {
        "cache_exists": cache_directory.exists(),
        "configured_remotes": remote_list or "none",
        "git_metadata": ["dataset.csv.dvc", "dvc.yaml", "dvc.lock"],
    }
)


## 4. Checks

data pointer, pipeline definition, lock state, 변경 감지와 재현 결과를 확인한 뒤 임시 project를 정리합니다.

In [ ]:
assert pointer_output["path"] == "dataset.csv"
assert len(pointer_output["md5"]) == 32
assert set(pipeline_document["stages"]["prepare"]) == {
    "cmd",
    "deps",
    "params",
    "outs",
}
assert first_result == 21
assert "prepare" in stale_status
assert "prepare" in dag_output
assert second_result == 35
assert "up to date" in final_status.lower()
assert cache_directory.exists()
temporary_directory.cleanup()
print("All appendix checks passed.")


## 5. Next Steps

- raw data, processed data와 split output을 DVC로 추적하고 source manifest와 role을 함께 기록합니다.
- stage가 실제로 사용하는 code, config, parameter와 input만 dependency로 선언합니다.
- `dvc repro` 후 `dvc status`를 확인하고 변경된 `dvc.lock`을 code와 함께 review합니다.
- remote credential은 repository 밖에서 관리하고 `dvc push`·`pull`의 접근 범위를 제한합니다.
- Git revision, DVC lock digest, MLflow Run과 release manifest를 서로 연결하되 역할을 하나로 합치지 않습니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [Get Started: Data Management](https://dvc.org/doc/start/data-management)
- [Get Started: Data Pipelines](https://dvc.org/doc/start/data-pipelines/data-pipelines)
- [dvc add](https://dvc.org/doc/command-reference/add)
- [dvc stage add](https://dvc.org/doc/command-reference/stage/add)
- [dvc repro](https://dvc.org/doc/command-reference/repro)
- [dvc status](https://dvc.org/doc/command-reference/status)
- [dvc dag](https://dvc.org/doc/command-reference/dag)
- [Remote Storage](https://dvc.org/doc/user-guide/data-management/remote-storage)